## 1 Intro / Введение

### 1.1 Пять примеров применения машинного обучения (ML, модель) в жизни
1. **Фильтрация спама в почте.** Модель автоматически отделяет полезные письма от нежелательных и экономит время пользователя.
2. **Кредитный скоринг в банке.** Модель оценивает вероятность возврата кредита и помогает банку управлять финансовыми рисками.
3. **Анализ медицинских снимков.** Модель может находить подозрительные области на снимках и помогать врачу быстрее поставить диагноз.
4. **Рекомендательные системы.** Модель подбирает фильмы, музыку, рецепты блюд, товары или курсы под интересы пользователя.
5. **Прогнозирование спроса в магазинах.** Модель помогает заранее оценить будущие продажи и правильно планировать запасы.

### 1.2 Классы задач
#### Примеры из таблицы теории можно отнести к таким классам:  
- 1 Предсказать цену дома —> РЕГРЕССИЯ. Обучение с учителем  
- 2 Предсказать, вернёт ли клиент кредит —> Бинарная КЛАССИФИКАЦИЯ
- 3 Предсказать, когда пациенту нужно принять лекарство —> РЕГРЕССИЯ - количество времени, которое должно пройти с момента последнего приёма лекарства с учителем
- 4 Выбрать лекарство для пациента —> Многоклассовая КЛАССИФИКАЦИЯ или рекомендательная задача 
- 5 Выбрать сегмент клиентов для акции —> КЛАССИФИКАЦИЯ, (целевое значение, например, участвует ли клиент в акции или нет, покупает ли товары по акции или КЛАСТЕРИЗАЦИЯ - определение и предсказание поведения групп клиентов. Обучение без учителя.
- 6 Распознать дефект по фото —> Бинарная КЛАССИФИКАЦИЯ целевое значение - наличие или отсутствие дефекта
- 7 Решить, как расставить товары на полке —> Регрессия/оптимизация, иногда reinforcement learning 
- 8 Поиск сайтов по текстовому запросу —> РАНЖИРОВАНИЕ, цель — не предсказать абсолютное значение или категорию, а упорядочить (отсортировать) список объектов так, чтобы наиболее полезные, релевантные или интересные объекты оказались в самом верху списка, а наименее полезные — внизу
- 9 Разделить покупателей на сегменты —> Кластеризация. Обучение без учителя. 
- 10 Обнаружить аномалию в трафике сайта —> Поиск аномалий, обучение без учителя или с частичным привлечением учителя
#### Мои пять примеров:
- 1 Фильтрация спама —> бинарная классификация
- 2 Кредитный скоринг —> бинарная классификация или регрессия
- 3 Анализ медицинских снимков —> классификация или сегментация
- 4 Рекомендательные системы —> ранжирование, классификация, ассоциативные правила
- 5 Прогноз спроса —> регрессия, прогнозирование временных рядов


### 1.3 Multiclass и multilabel

В **многоклассовой классификации** объект относится ровно к одному классу из нескольких возможных. Например, фрукт может быть яблоком, грушей или бананом.

В **многометочной классификации** объект может иметь несколько меток одновременно. Например, фильм может быть и комедией, и мелодрамой.

### 1.4 Ответ на вопрос
#### Вопрос:
Является ли пример с ценами на дома из теории задачей классификации или регрессии? Можно ли свести задачу регрессии к классификации?
#### Ответ:
Пример с ценами на дома — это регрессия, потому что целевая переменная является непрерывным числом. Регрессию можно свести к классификации, если разбить цены на группы, например, дешёвые, средние и дорогие. Но при таком подходе теряется часть информации: точные цены заменяются категориями.

## 2 Введение в анализ данных

In [11]:
%time
# Импортируем библиотеки для анализа данных, визуализации и обучения моделей.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import lightgbm as lgb



warnings.filterwarnings("ignore") # Скрыть предупреждения
sns.set_theme(style="whitegrid", palette="Set2") # Сразу настроим фон и палитру для графиков
pd.set_option("display.max_columns", 100) # — чтобы пандас все столбцы показывал всегда (до сотни)
print('все библиотеки импортированы')

CPU times: total: 0 ns
Wall time: 5.48 μs
все библиотеки импортированы


In [14]:
%time
def find_train_json():
    candidates = [
        Path("src/train.json"),
        Path("datasets/train.json"),
        Path("data/train.json"),
        Path("train.json"),
        Path("../src/train.json"),
        Path("../datasets/train.json"),
        Path("../data/train.json"),
        Path("../train.json"),
    ]
    for path in candidates:
        if path.exists():
            print(path)
            return path
    raise FileNotFoundError(
        "train.json was not found. Download it from Kaggle and place it into datasets/train.json "
        "or data/train.json."
    )


DATA_PATH = find_train_json()
data = pd.read_json(DATA_PATH)
data.head()

CPU times: total: 0 ns
Wall time: 5.25 μs
train.json


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low
